# ViANLI-RL: SFT + GRPO với Suy Luận Căn Chỉnh Tường Minh
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baonguyen-infina/rl-nli-/blob/master/notebooks/g0_setup.ipynb)

Notebook này thiết lập môi trường và chạy các thí nghiệm giai đoạn **G0** (sanity check) và **G1** (baselines & probe) theo spec nghiên cứu ViANLI.

### Lộ trình thực thi:
1. **Thiết lập môi trường**: GPU check, clone repo, mount Drive (tùy chọn), cài đặt dependencies.
2. **Dữ liệu & Test**: Chuẩn bị dataset ViANLI (8,012 train, 1,000 dev, 300 dev-mini), sinh config ma trận, chạy unit tests (76 tests).
3. **Cổng G0**: Chạy thử nghiệm 1 bước GRPO (`G0`) để xác nhận luồng trainer + reward function.
4. **Cổng G1**: Chạy các baseline:
   - Zero-shot LLM: `E0` (label), `E1` (CoT), `E2` (align)
   - SFT LLM: `E3` (label)
   - Encoder: `B1` (CafeBERT)
   - Hypothesis-only probe: 5-fold cross-fitting
   - Kiểm tra điều kiện qua cổng: `check_g1` ($E3 > \max(E0..E2)$ & parse $\ge 99\%$).
5. **Tổng hợp kết quả**: Xem bảng so sánh metrics và tự động sao lưu vào Drive.

## 1. Kiểm tra GPU và Cấu hình Môi trường

In [ ]:
# 1.1 Kiểm tra thông tin GPU
!nvidia-smi

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Mount Google Drive (Tùy chọn) để sao lưu kết quả

In [ ]:
# 2.1 Mount Google Drive (để lưu giữ kết quả khi phiên làm việc Colab kết thúc)
import os
USE_DRIVE = True
drive_backup_dir = "/content/drive/MyDrive/rl-nli_backup"

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(f"{drive_backup_dir}/results", exist_ok=True)
        os.makedirs(f"{drive_backup_dir}/artifacts", exist_ok=True)
        print(f"✅ Đã mount Drive thành công. Bản sao lưu sẽ tại: {drive_backup_dir}")
    except Exception as e:
        print(f"⚠️ Bỏ qua mount Drive (kết quả lưu cục bộ tại máy ảo Colab): {e}")

## 3. Clone Repository & Cài đặt Dependencies

In [ ]:
# 3.1 Clone mã nguồn từ GitHub
import os
repo_url = "https://github.com/baonguyen-infina/rl-nli-.git"
repo_dir = "/content/rl-nli"

if not os.path.exists(repo_dir):
    !git clone {repo_url} {repo_dir}
else:
    !cd {repo_dir} && git pull

%cd {repo_dir}

In [ ]:
# 3.2 Cài đặt các thư viện cần thiết
# Gỡ torchao (bản 0.10.0 cũ cài sẵn trong Colab gây xung đột với peft LoRA)
!pip uninstall -y torchao || true
!pip install -q -r requirements.txt


## 4. Chuẩn bị Dữ liệu ViANLI & Chạy Unit Tests

In [ ]:
# 4.1 Tải & chuẩn hoá dataset ViANLI từ HuggingFace; sinh configs cho các runs
!python -m data.prepare
!python -m configs.matrix

In [ ]:
# 4.2 Chạy bộ test suite (76 unit tests - đảm bảo parser, reward, metrics, logic hoạt động 100%)
!python -m pytest -q

## 5. Cổng G0: Sanity Check 1 bước GRPO
Kiểm tra toàn bộ luồng huấn luyện TRL GRPOTrainer + vLLM rollout + hàm thưởng với model Qwen3.5-2B.

In [ ]:
# 5.1 Chạy 1 bước GRPO (G0)
!python -m scripts.run --run G0 --seed 0

## 6. Giai đoạn G1: Zero-shot Baselines (E0, E1, E2)
Sử dụng vLLM greedy decoding trên dev split:
- `E0`: Zero-shot nhãn đơn thuần `<answer>label</answer>`
- `E1`: Zero-shot Chain-of-Thought `<think>...</think><answer>...</answer>`
- `E2`: Zero-shot căn chỉnh trực tiếp `<align>...</align><answer>...</answer>`

In [ ]:
# 6.1 Chạy Zero-shot runs
!python -m scripts.run --run E0 --seed 0
!python -m scripts.run --run E1 --seed 0
!python -m scripts.run --run E2 --seed 0

## 7. Giai đoạn G1: SFT Baseline (E3)
Huấn luyện Supervised Fine-Tuning với LoRA trên ViANLI train (8,012 mẫu) ở format nhãn (`E3`), tự động đánh giá từng epoch trên dev và chọn checkpoint tốt nhất.

In [ ]:
# 7.1 Chạy SFT E3
!python -m scripts.run --run E3 --seed 0

## 8. Giai đoạn G1: Encoder Baseline (B1 - CafeBERT) & Hypothesis Probe
- `B1`: Fine-tune `uitnlp/CafeBERT` trên premise + hypothesis để làm mốc tham chiếu so sánh.
- `hyp_probe`: 5-fold cross-fitting CafeBERT chỉ trên hypothesis để đo lường mức độ rò rỉ giả thuyết (hypothesis bias).

In [ ]:
# 8.1 Fine-tune B1 (CafeBERT NLI)
!python -m scripts.run --run B1 --seed 0

In [ ]:
# 8.2 Chạy Hypothesis-only 5-fold cross-fitting probe
!python -m data.hyp_probe

## 9. Kiểm Tra Điều Kiện Cổng G1 (`check_g1`)
Cổng G1 đạt yêu cầu khi:
1. Đã có đủ kết quả cho `E0`, `E1`, `E2`, `E3`, `B1` và `hyp_probe`.
2. $E3 \text{ (SFT)} > \max(E0, E1, E2) \text{ (Zero-Shot)}$ trên dev split.
3. Tỷ lệ parse thành công của $E3 \ge 99\%$.

In [ ]:
# 9.1 Chạy script kiểm tra cổng G1
!python -m scripts.check_g1

## 10. Tổng Hợp Kết Quả & Sao Lưu Vào Google Drive

In [ ]:
# 10.1 Hiển thị bảng tổng hợp kết quả metrics
import json
from pathlib import Path

results_dir = Path("results")
print(f"{'Run ID':<10} | {'Model':<15} | {'Accuracy':<10} | {'Macro F1':<10} | {'Parse Rate':<12}")
print("-" * 68)
for m_file in sorted(results_dir.glob("*/metrics_dev.json")):
    run_name = m_file.parent.name
    with open(m_file, "r", encoding="utf-8") as f:
        m = json.load(f)
    acc = f"{m.get('accuracy', 0):.4f}"
    f1 = f"{m.get('macro_f1', 0):.4f}"
    parse = f"{m.get('parse_rate', 0):.4f}" if "parse_rate" in m else "N/A"
    print(f"{run_name.split('_')[0]:<10} | {run_name:<15} | {acc:<10} | {f1:<10} | {parse:<12}")

In [ ]:
# 10.2 Đồng bộ artifacts và results sang Google Drive (nếu đã mount)
!cp -ru /content/rl-nli/results /content/drive/MyDrive/rl-nli_backup/ 2>/dev/null && echo "✅ Đã backup results sang Google Drive!" || echo "(Không backup do chưa mount Drive)"
!cp -ru /content/rl-nli/artifacts /content/drive/MyDrive/rl-nli_backup/ 2>/dev/null && echo "✅ Đã backup artifacts sang Google Drive!" || true